In [1]:
# =====================================
# ✅ 1. Install Required Libraries
# =====================================
!pip install transformers torch --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.1 MB/s eta 0:00:00


In [2]:
# =====================================
# ✅ 2. Import Libraries
# =====================================
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch


In [3]:
# =====================================
# ✅ 3. Load the Pretrained GPT-2 Model
# =====================================
# Model options: "gpt2", "gpt2-medium", "gpt2-large", "gpt2-xl"
model_name = "gpt2"

# Load tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Set model to evaluation mode (inference mode)
model.eval()


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
# =====================================
# ✅ 4. (Optional) Check GPU Availability
# =====================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Using device:", device)


Using device: cpu


In [7]:
# =====================================
# ✅ 5. Input Prompt and Encode Text
# =====================================
prompt = "Once upon a time in a distant galaxy"

# Set padding token if it's not already set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
attention_mask = input_ids.ne(tokenizer.pad_token_id).long().to(device)

In [8]:
# =====================================
# ✅ 6. Generate Text from GPT-2
# =====================================
# Generation parameters:
# - max_length: Total length (prompt + generated)
# - temperature: Randomness (higher = more creative)
# - top_k and top_p: Sampling controls

output_ids = model.generate(
    input_ids,
    attention_mask=attention_mask, # Added attention mask
    pad_token_id=tokenizer.eos_token_id, # Set pad token id
    max_length=100,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    do_sample=True,
    no_repeat_ngram_size=2,
    num_return_sequences=1
)

# Decode output to readable text
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("📜 Generated Text:\n")
print(generated_text)

📜 Generated Text:

Once upon a time in a distant galaxy, a small group of astronomers discovered the first known dwarf star. It was discovered in the constellation Draco, named after the planet's famous star in ancient Greek mythology.

While the star was in its infancy, scientists discovered that it had a very small star, called a Tetragon. The Teterboro-based team believed that the Tertiary star had formed after a collision with a star from the distant universe known as the Big Bang, which
